# Recommender system: SVD and beyond

In [1]:
import numpy as np
import pandas as pd

## Simulated movie rating data
- each row is a user
- each column is a movie
- some ratings are unknown 

**Goal: guess the ratings for unrated movies**

In [4]:
R = np.full([8,11], fill_value=np.nan)
R[:4,:]= np.array([[4.5,4.5,4.5,2,3.5,3,5,2.5,4.5,5,4],
                    [4,4,4.5,1.5,4,3.5,5,2,4.5,5,4],
                    [5,5,3.5,4,2.5,2,4,5,4.5,4,3.5],
                    [4,5,4,2.5,3,3,4.5,3,5,4.5,3.5]])
R[4,1]=5;R[4,9]=3;
R[5,2]=5;R[5,6]=4.5;
R[6,3]=2;R[6,10]=4.5;
R[7,4]=3.5; R[7,6]=2;
R
R_df = pd.DataFrame(R, columns= ['E.T.',
                                   'Dracula', 
                                   'Die Hard', 
                                   'The Room',
                                   'Love, actually',
                                   'No Time to Die',
                                   'Lion King',
                                   'Top Gun',
                                   'Brave Heart',
                                   'Fifth Element',
                                   'True Romance',])
R_df

,E.T.,Dracula,Die Hard,The Room,"Love, actually",No Time to Die,Lion King,Top Gun,Brave Heart,Fifth Element,True Romance
0,4.5,4.5,4.5,2.0,3.5,3.0,5.0,2.5,4.5,5.0,4.0
1,4.0,4.0,4.5,1.5,4.0,3.5,5.0,2.0,4.5,5.0,4.0
2,5.0,5.0,3.5,4.0,2.5,2.0,4.0,5.0,4.5,4.0,3.5
3,4.0,5.0,4.0,2.5,3.0,3.0,4.5,3.0,5.0,4.5,3.5
4,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
5,NaN,NaN,5.0,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,4.5
7,NaN,NaN,NaN,NaN,3.5,NaN,2.0,NaN,NaN,NaN,NaN


## Create a mask to indicate known ratings
- known entry = 1
- unknown entry = 0

In [11]:
mask = np.ones(R.shape)
mask[np.isnan(R)]=0
mask = mask.astype(bool)
mask

array([[ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True],
       [False,  True, False, False, False, False, False, False, False,
         True, False],
       [False, False,  True, False, False, False,  True, False, False,
        False, False],
       [False, False, False,  True, False, False, False, False, False,
        False,  True],
       [False, False, False, False,  True, False,  True, False, False,
        False, False]])

## Main Idea
*"Perception of the whole is based on perception of its parts."*

The ratings, collectively, has a low-dimensional structure. This means **the rating matrix is approximately low rank**.

Here is a very simplified example: we assume users are rating movies based on only **two features**

- User 0 decides how important each feature is to them: 

||Feature 1 (story) |Feature 2 (action)|
|:--|--|--|
|user 0|2/3 | 1/3|

- "General concensus" on how each movie scores on these features:

||E.T.|Dracula|
|:--|:--|--|
|Feature 1 (story)|4.5|6|
|Feature 2 (action)|4.5|1.5|

- Final rating
  - user 0's rating for *E.T.*: $\begin{bmatrix}2/3&1/3\end{bmatrix}\begin{bmatrix}
4.5\\
4.5
\end{bmatrix}=4.5$

$$R \approx \begin{bmatrix}2/3&1/3\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?\\
?&?
\end{bmatrix}\begin{bmatrix}
4.5&6&?&?&?&?&?&?&?&?&?\\
4.5&1.5&?&?&?&?&?&?&?&?&?
\end{bmatrix}=PQ$$


## Initial simple imputation
This is often done by filling with averages of each movie (column)

In [6]:
np.set_printoptions(precision = 2, suppress = True)

In [7]:
# compute average (with NA's excluded)
col_avg = np.nanmean(R, axis=0) # along axis 0 (rows), so col mean
R0 = R.copy()
for j in range(R.shape[1]):
    R0[np.isnan(R[:,j]),j] = col_avg[j]
print(R0)

[[4.5  4.5  4.5  2.   3.5  3.   5.   2.5  4.5  5.   4.  ]
 [4.   4.   4.5  1.5  4.   3.5  5.   2.   4.5  5.   4.  ]
 [5.   5.   3.5  4.   2.5  2.   4.   5.   4.5  4.   3.5 ]
 [4.   5.   4.   2.5  3.   3.   4.5  3.   5.   4.5  3.5 ]
 [4.38 5.   4.3  2.4  3.3  2.88 4.17 3.12 4.62 3.   3.9 ]
 [4.38 4.7  5.   2.4  3.3  2.88 4.5  3.12 4.62 4.3  3.9 ]
 [4.38 4.7  4.3  2.   3.3  2.88 4.17 3.12 4.62 4.3  4.5 ]
 [4.38 4.7  4.3  2.4  3.5  2.88 2.   3.12 4.62 4.3  3.9 ]]


## Method 1: plain SVD

In [8]:
U, S, Vt = np.linalg.svd(R0, full_matrices=False)
V = Vt.T
k = 2 # can be adjusted
Sm = np.diag(S[:k])
Rh1 = U[:,:k]@Sm@(V[:,:k]).T # this is rank-k approximation of R0
for x, y in zip(Rh1, R):
    print(f"{x} \t {y}")

[4.29 4.6  4.64 1.91 3.68 3.25 4.64 2.54 4.73 4.69 4.12] 	 [4.5 4.5 4.5 2.  3.5 3.  5.  2.5 4.5 5.  4. ]
[3.99 4.27 4.72 1.35 3.84 3.44 4.83 1.86 4.59 4.81 4.11] 	 [4.  4.  4.5 1.5 4.  3.5 5.  2.  4.5 5.  4. ]
[4.94 5.32 3.78 3.82 2.61 2.15 3.37 4.84 4.72 3.68 3.66] 	 [5.  5.  3.5 4.  2.5 2.  4.  5.  4.5 4.  3.5]
[4.37 4.69 4.31 2.39 3.31 2.88 4.19 3.11 4.62 4.31 3.9 ] 	 [4.  5.  4.  2.5 3.  3.  4.5 3.  5.  4.5 3.5]
[4.33 4.65 4.12 2.52 3.12 2.71 3.97 3.26 4.51 4.11 3.77] 	 [nan  5. nan nan nan nan nan nan nan  3. nan]
[4.45 4.77 4.46 2.35 3.44 3.01 4.36 3.07 4.74 4.47 4.02] 	 [nan nan 5.  nan nan nan 4.5 nan nan nan nan]
[4.36 4.68 4.38 2.3  3.39 2.96 4.29 3.   4.65 4.4  3.95] 	 [nan nan nan 2.  nan nan nan nan nan nan 4.5]
[4.27 4.58 3.97 2.57 2.98 2.57 3.79 3.32 4.4  3.95 3.65] 	 [nan nan nan nan 3.5 nan 2.  nan nan nan nan]


In [ ]:
# What is P, Q here?

On the known ratings, Rh1 has different entries than R

...

In [9]:
def error_known(filled, given, mask):
    e = np.linalg.norm(filled*mask - given*mask, 'fro')
    e = e**2
    e = e/mask.sum()
    return(e**0.5)

In [21]:
# error
error_known(Rh1, R0, mask)

np.float64(0.39326651742037017)

In [22]:
# fill back the known ratings
Rh1_final = Rh1.copy()
Rh1_final[mask] = R0[mask]
Rh1_final

array([[4.5 , 4.5 , 4.5 , 2.  , 3.5 , 3.  , 5.  , 2.5 , 4.5 , 5.  , 4.  ],
       [4.  , 4.  , 4.5 , 1.5 , 4.  , 3.5 , 5.  , 2.  , 4.5 , 5.  , 4.  ],
       [5.  , 5.  , 3.5 , 4.  , 2.5 , 2.  , 4.  , 5.  , 4.5 , 4.  , 3.5 ],
       [4.  , 5.  , 4.  , 2.5 , 3.  , 3.  , 4.5 , 3.  , 5.  , 4.5 , 3.5 ],
       [4.33, 5.  , 4.12, 2.52, 3.12, 2.71, 3.97, 3.26, 4.51, 3.  , 3.77],
       [4.45, 4.77, 5.  , 2.35, 3.44, 3.01, 4.5 , 3.07, 4.74, 4.47, 4.02],
       [4.36, 4.68, 4.38, 2.  , 3.39, 2.96, 4.29, 3.  , 4.65, 4.4 , 4.5 ],
       [4.27, 4.58, 3.97, 2.57, 3.5 , 2.57, 2.  , 3.32, 4.4 , 3.95, 3.65]])

## Method 2: SVD - with centering
$R\approx  PQ + $ col_avg


In [17]:
col_avg = np.mean(R0, axis=0)
R0_c = R0 - col_avg
U,S,Vt = np.linalg.svd(R0_c)
V = Vt.T
k = 2
Sm = np.diag(S[:k])
# Rk is the rank-k approximation of R0_m
Rh2 = U[:,:k]@Sm@(V[:,:k]).T+col_avg
error = error_known(Rh2, R0, mask)
print(Rh2)
print(error)

[[4.22 4.5  4.49 1.95 3.51 3.14 5.07 2.56 4.62 4.64 3.97]
 [4.03 4.31 4.74 1.41 3.83 3.44 5.04 1.92 4.62 4.85 4.12]
 [4.91 5.23 3.62 3.93 2.4  2.03 4.06 4.92 4.63 3.67 3.51]
 [4.4  4.7  4.27 2.46 3.24 2.85 4.55 3.18 4.62 4.34 3.87]
 [4.43 4.77 4.23 2.56 3.22 2.78 3.94 3.33 4.63 4.19 3.87]
 [4.35 4.66 4.33 2.33 3.32 2.92 4.48 3.02 4.62 4.39 3.91]
 [4.32 4.65 4.37 2.25 3.39 2.96 4.12 2.95 4.63 4.35 3.94]
 [4.34 4.78 4.35 2.31 3.49 2.89 2.07 3.12 4.65 3.96 4.02]]
0.26934398330811743


## Method 3: Nonnegative Matrix Factorization
Non-negative Matrix Factorization is to write a given matrix (with all positive entries) as product of two low-rank matrices, **both with nonnegative entries**:
$$R\approx PQ, \qquad P\in\mathbb{R}^{m\times k}, Q\in\mathbb{R}^{k\times n}, P_{ij}\geq0, Q_{ij}\geq0$$
$P$ and $Q$ are **low-rank** as $k\ll m, k\ll n$ 

Applications:
- originated in chemometrics
- genomics
- computer vision
- natural language processing (topic modeling)
- recommender system
- ...


In [26]:
max_iter = 400
m, n = R0.shape
R = R0*mask
np.random.seed(3)
P = np.random.rand(m, k)
Q = np.random.rand(k, n)
eta = 0.01
for i in range(max_iter):
    PQ = (P@Q)*mask
    P = P + eta*(R@Q.T - PQ@Q.T) # P = P - eta * dP
    # problem : no guarantee that P, Q will have nonneg entries
    # potential solution: add a line to let P, Q go through relu()
    Q = Q + eta*(P.T@R - P.T@PQ) # Q = Q - etaa * dQ

In [27]:
R_nmf = P@Q
error = error_known(R_nmf, R0, mask)
print(error)

0.21445343010779488


In [28]:
P

array([[ 1.13,  1.84],
       [ 0.76,  2.04],
       [ 2.31,  0.9 ],
       [ 1.43,  1.54],
       [ 2.27,  0.61],
       [ 0.55,  2.21],
       [ 1.04,  2.23],
       [-0.19,  1.51]])

In [29]:
Q

array([[1.57, 1.75, 0.72, 1.64, 0.18, 0.29, 1.14, 2.01, 1.32, 0.81, 0.85],
       [1.38, 1.39, 2.  , 0.12, 1.94, 1.56, 1.83, 0.19, 1.75, 2.18, 1.62]])